# 실험 ② — 데이터 추가 ablation (Colab)

> **근거**: `docs/감정분류_개선실험_계획서_김한솔.md` §4 · 전제: **실험 ① 완료 후 승자 조합을 아래 CONFIG에 기입**
> **원칙**: 시험지(작성체 test + 채팅체 eval)는 절대 고정 — 추가 데이터는 **학습에만** 합류.
> 오르는 것만 채택, "추가하면 잘될 것"이라는 가정 금지.

| 조합 | 학습 데이터 |
|---|---|
| base | 감성대화 58,234 |
| +voice | base + 음성 전사 36,677 (5인 중 3표 합의) |
| +wellness | base + 웰니스 상담 발화 1,002 (⚠ 기쁨 12건) |
| +KOTE | base + 한국어 댓글 (HuggingFace에서 자동 다운로드) |

**준비물 업로드**: `kcelectra_train_clean.jsonl`, `chat_eval_set.jsonl`, `aug_voice.jsonl`, `aug_wellness.jsonl`
(aug 파일은 `ai/emotion/prepare_aug_datasets.py`로 생성)

In [ ]:
!pip install -q transformers==4.44.2 xgboost scikit-learn sentencepiece
!pip install -q 'git+https://github.com/SKTBrain/KoBERT.git#egg=kobert_tokenizer&subdirectory=kobert_hf'

In [ ]:
# ===== CONFIG — 실험 ① 결과를 여기 기입 =====
WINNER_MODEL = 'KcELECTRA'      # 'KcELECTRA' | 'KoBERT'
WINNER_METHOD = 'finetune'      # 'finetune' | 'frozen'
WINNER_RECIPE = 'last4_mean'    # frozen일 때만 사용
QUICK_MODE = True               # True: base를 1.2만으로 축소해 파이프라인 검증
# ============================================

import json, random
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

SEED, MAX_LEN, QUICK_N = 42, 128, 12000
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
EMO4 = ['기쁨', '슬픔', '분노', '일반']
LAB2ID = {c: i for i, c in enumerate(EMO4)}
SIX_TO_FOUR = {'기쁨':'기쁨','슬픔':'슬픔','상처':'슬픔','분노':'분노','불안':'분노','당황':'일반'}

def load_jsonl(path, mapping=None):
    rows = [json.loads(l) for l in open(path, encoding='utf-8') if l.strip()]
    d = pd.DataFrame(rows)
    d['label'] = d['emotion'].map(mapping) if mapping else d['emotion']
    return d[d['label'].isin(EMO4)][['text', 'label']].reset_index(drop=True)

base = load_jsonl('kcelectra_train_clean.jsonl', SIX_TO_FOUR)
if QUICK_MODE:
    base = base.groupby('label', group_keys=False).apply(
        lambda g: g.sample(min(len(g), QUICK_N // 4), random_state=SEED)).reset_index(drop=True)
base['y'] = base['label'].map(LAB2ID)

# 시험지 고정 (base에서만 분리 — 추가 데이터는 절대 여기 안 들어감)
tr_df, te_df = train_test_split(base, test_size=0.2, random_state=SEED, stratify=base['y'])
te_texts, te_y = te_df['text'].tolist(), te_df['y'].values
chat = pd.DataFrame([json.loads(l) for l in open('chat_eval_set.jsonl', encoding='utf-8')])
chat_texts, chat_y = chat['text'].tolist(), chat['emotion'].map(LAB2ID).values

aug_voice = load_jsonl('aug_voice.jsonl')
aug_wellness = load_jsonl('aug_wellness.jsonl')
print(f'base train {len(tr_df):,} / test {len(te_texts):,} / chat {len(chat_texts)}')
print(f'aug: voice {len(aug_voice):,} · wellness {len(aug_wellness):,}')

## KOTE 로드 — GitHub TSV 직접 로드 + 명시 매핑

KOTE는 44개 세부 감정의 다중 라벨. **딱 하나의 4모드 버킷으로만 매핑되는 문장**만 채택해
라벨 모호성을 차단한다 (여러 버킷에 걸치면 제외).

In [ ]:
# KOTE — GitHub 원본 TSV 직접 로드 (datasets 라이브러리 불필요 · 스크립트 로딩 문제 원천 회피)
KOTE_LABELS = ['불평/불만','환영/호의','감동/감탄','지긋지긋','고마움','슬픔','화남/분노','존경',
 '기대감','우쭐댐/무시함','안타까움/실망','비장함','의심/불신','뿌듯함','편안/쾌적','신기함/관심',
 '아껴주는','부끄러움','공포/무서움','절망','한심함','역겨움/징그러움','짜증','어이없음','없음',
 '패배/자기혐오','귀찮음','힘듦/지침','즐거움/신남','깨달음','죄책감','증오/혐오','흐뭇함(귀여움/예쁨)',
 '당황/난처','경악','부담/안_내킴','서러움','재미없음','불쌍함/연민','놀람','행복','불안/걱정','기쁨','안심/신뢰']

# 44 → 4모드 명시 매핑 (감성대화 규칙과 일관: 불안·공포→분노, 당황·놀람→일반)
# 미포함 라벨 = 감정 방향이 모호하거나 타인 지향(연민 등) — 버킷 판정에서 무시
KOTE_TO_4 = {
    '기쁨': ['환영/호의','감동/감탄','고마움','존경','기대감','뿌듯함','편안/쾌적',
            '즐거움/신남','흐뭇함(귀여움/예쁨)','행복','기쁨','안심/신뢰'],
    '슬픔': ['슬픔','안타까움/실망','절망','패배/자기혐오','힘듦/지침','서러움','죄책감'],
    '분노': ['불평/불만','지긋지긋','화남/분노','한심함','역겨움/징그러움','짜증','어이없음',
            '증오/혐오','공포/무서움','불안/걱정'],
    '일반': ['없음','당황/난처','놀람'],
}
idx2bucket = {}
for bucket, names in KOTE_TO_4.items():
    for n in names:
        idx2bucket[KOTE_LABELS.index(n)] = bucket

try:
    urls = [f'https://raw.githubusercontent.com/searle-j/KOTE/main/{s}.tsv'
            for s in ['train', 'val', 'test']]
    kote_df = pd.concat([pd.read_csv(u, sep='\t', names=['ID', 'text', 'labels'])
                         for u in urls], ignore_index=True)
    rows = []
    for _, r in kote_df.iterrows():
        try:
            idxs = [int(x) for x in str(r['labels']).split(',')]
        except ValueError:
            continue
        buckets = {idx2bucket[i] for i in idxs if i in idx2bucket}
        if len(buckets) == 1:                     # 단일 버킷 수렴 문장만 (라벨 모호성 차단)
            t = str(r['text']).strip()
            if 2 <= len(t) <= 300:
                rows.append({'text': t, 'label': buckets.pop()})
    aug_kote = pd.DataFrame(rows)
    print(f'KOTE {len(kote_df):,}건 → 단일 버킷 수렴 {len(aug_kote):,}건 채택')
    print('분포:', aug_kote.label.value_counts().to_dict())
except Exception as e:
    aug_kote = pd.DataFrame(columns=['text', 'label'])
    print(f'KOTE 로드 실패 — 이 조합은 건너뜀: {e}')

## 승자 조합 학습기 (실험 ①과 동일 코드)

In [ ]:
MODELS = {'KcELECTRA': 'beomi/KcELECTRA-base-v2022', 'KoBERT': 'skt/kobert-base-v1'}

def load_tokenizer(key):
    if key == 'KoBERT':
        from kobert_tokenizer import KoBERTTokenizer
        return KoBERTTokenizer.from_pretrained(MODELS[key])
    from transformers import AutoTokenizer
    return AutoTokenizer.from_pretrained(MODELS[key])

def score(y, p):
    return {'acc': accuracy_score(y, p), 'f1': f1_score(y, p, average='macro')}

@torch.no_grad()
def embed(texts, tok, enc, recipe='last4_mean', bs=64):
    embs = []
    for i in range(0, len(texts), bs):
        e = tok(texts[i:i+bs], padding=True, truncation=True, max_length=MAX_LEN,
                return_tensors='pt').to(device)
        hs = enc(**e).hidden_states
        mask = e['attention_mask'].unsqueeze(-1).float()
        pool = lambda h: (h * mask).sum(1) / mask.sum(1).clamp(min=1)
        if recipe == 'last4_mean':
            v = torch.cat([pool(h) for h in hs[-4:]], -1)
        elif recipe == 'last1_mean':
            v = pool(hs[-1])
        elif recipe == 'last4_cls':
            v = torch.cat([h[:, 0] for h in hs[-4:]], -1)
        else:
            v = hs[-1][:, 0]
        embs.append(v.cpu().numpy())
    emb = np.concatenate(embs)
    emb /= (np.linalg.norm(emb, axis=1, keepdims=True) + 1e-9)
    return emb

def run_frozen(train_texts, train_y):
    from transformers import AutoModel
    from xgboost import XGBClassifier
    tok = load_tokenizer(WINNER_MODEL)
    enc = AutoModel.from_pretrained(MODELS[WINNER_MODEL], output_hidden_states=True).eval().to(device)
    Xtr = embed(train_texts, tok, enc, WINNER_RECIPE)
    Xte = embed(te_texts, tok, enc, WINNER_RECIPE)
    Xch = embed(chat_texts, tok, enc, WINNER_RECIPE)
    del enc; torch.cuda.empty_cache()
    freq = pd.Series(train_y).value_counts()
    w = np.array([len(train_y) / (len(EMO4) * freq[c]) for c in train_y])
    clf = XGBClassifier(n_estimators=600, max_depth=6, learning_rate=0.08,
                        tree_method='hist', device=device, random_state=SEED)
    clf.fit(Xtr, train_y, sample_weight=w)
    return score(te_y, clf.predict(Xte)), score(chat_y, clf.predict(Xch))

def run_finetune(train_texts, train_y, epochs=3, lr=2e-5, bs=32):
    from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
    tok = load_tokenizer(WINNER_MODEL)

    class WT(Trainer):
        def __init__(self, class_weights=None, **kw):
            super().__init__(**kw); self.cw = class_weights
        def compute_loss(self, model, inputs, return_outputs=False, **kw):
            labels = inputs.pop('labels')
            out = model(**inputs)
            loss = torch.nn.functional.cross_entropy(out.logits, labels, weight=self.cw)
            return (loss, out) if return_outputs else loss

    class DS(torch.utils.data.Dataset):
        def __init__(self, texts, y):
            self.enc = tok(list(texts), padding=True, truncation=True, max_length=MAX_LEN)
            self.y = y
        def __len__(self): return len(self.y)
        def __getitem__(self, i):
            item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
            item['labels'] = torch.tensor(int(self.y[i])); return item

    model = AutoModelForSequenceClassification.from_pretrained(
        MODELS[WINNER_MODEL], num_labels=len(EMO4)).to(device)
    freq = pd.Series(train_y).value_counts().sort_index()
    cw = torch.tensor((len(train_y) / (len(EMO4) * freq)).values, dtype=torch.float).to(device)
    args = TrainingArguments(output_dir='/tmp/ft2', num_train_epochs=epochs, learning_rate=lr,
        per_device_train_batch_size=bs, per_device_eval_batch_size=64,
        fp16=(device == 'cuda'), logging_steps=200, seed=SEED,
        save_strategy='no', report_to='none')
    trainer = WT(class_weights=cw, model=model, args=args, train_dataset=DS(train_texts, train_y))
    trainer.train()
    p_test = trainer.predict(DS(te_texts, te_y)).predictions.argmax(-1)
    p_chat = trainer.predict(DS(chat_texts, chat_y)).predictions.argmax(-1)
    del trainer, model; torch.cuda.empty_cache()
    return score(te_y, p_test), score(chat_y, p_chat)

RUN = run_finetune if WINNER_METHOD == 'finetune' else run_frozen

## ablation 실행 — base부터 하나씩 추가

In [ ]:
combos = {'base': None, '+voice': aug_voice, '+wellness': aug_wellness, '+KOTE': aug_kote}
results = {}
for name, aug in combos.items():
    if aug is not None and len(aug) == 0:
        print(f'[{name}] 데이터 없음 — 건너뜀'); continue
    train = tr_df[['text', 'label']].copy()
    if aug is not None:
        train = pd.concat([train, aug[['text', 'label']]], ignore_index=True)
        train = train.drop_duplicates(subset='text')          # 셋 간 중복 문장 제거
    train_y = train['label'].map(LAB2ID).values
    print(f'===== {name}: 학습 {len(train):,}건 =====')
    t, c = RUN(train['text'].tolist(), train_y)
    results[name] = {'작성체 F1': t['f1'], '작성체 Acc': t['acc'],
                     '채팅체 F1': c['f1'], '채팅체 Acc': c['acc']}
    print(f"  작성체 F1 {t['f1']:.4f} / 채팅체 F1 {c['f1']:.4f}")

## 판정 — base 대비 오른 것만 채택

In [ ]:
tbl = pd.DataFrame(results).T.round(4)
display(tbl)
b = tbl.loc['base']
print('\n[base 대비 변화]')
for name in tbl.index:
    if name == 'base':
        continue
    d_test = tbl.loc[name, '작성체 F1'] - b['작성체 F1']
    d_chat = tbl.loc[name, '채팅체 F1'] - b['채팅체 F1']
    # 판정: ① 작성체 상승 + 채팅체 비하락 → 채택
    #        ② 작성체 하락이 노이즈 범위(-0.01 이내)인데 채팅체가 크게 상승(+0.05↑) → 채택 후보
    #           (실사용 문체 성능을 우선하되, 60문장 시험지라 본실험+증량 재확인 필수)
    if d_test > 0 and d_chat >= 0:
        verdict = '✅ 채택 후보'
    elif d_test > -0.01 and d_chat >= 0.05:
        verdict = '✅ 채택 후보 (채팅체 우세 — 평가셋 증량 후 재확인)'
    else:
        verdict = '❌ 기각 (base 유지)'
    print(f'  {name}: 작성체 {d_test:+.4f} / 채팅체 {d_chat:+.4f} → {verdict}')

with open('exp2_results.json', 'w', encoding='utf-8') as f:
    json.dump(tbl.to_dict('index'), f, ensure_ascii=False, indent=2)
print('\nexp2_results.json 저장 → repo 커밋 + 계획서 §4 기입')

## 해석 가이드

- **채택 기준**: 작성체 F1 상승 **그리고** 채팅체 F1 비하락 — 한쪽을 희생하는 추가는 기각.
- 채택 후보가 여러 개면 → 채택분끼리 합친 조합(+voice+KOTE 등)을 마지막에 한 번 더.
- 웰니스가 기각되어도 정상 — 기쁨 12건짜리 불균형 셋이라 예상된 결과 (그래서 실험으로 확인한 것).
- 최종 채택 조합으로 전체 재학습 → 산출물 갱신(`artifacts/`) → 서비스 배포는 `emotion_model.py` 경로 그대로.

## 최종 학습 — 승자 조합으로 배포용 모델 생성 + 150문장 확정 평가 + 저장

판정표를 보고 `FINAL_AUG`만 정해서 실행. 하는 일 세 가지:
① 승자 조합 전체 학습(배포용) ② **증량 평가셋(150문장) 확정 채점** ③ 모델을 Drive에 저장

⚠ 실행 전: 검수 끝난 `chat_eval_set.jsonl`(150문장)을 Drive `final-test` 폴더에 **덮어쓰기 업로드**할 것.

In [ ]:
# ===== 최종 학습 CONFIG =====
FINAL_AUG = '+voice'        # 판정 결과 보고 결정: 'base' | '+voice' | '+KOTE' | '+voice+KOTE'
SAVE_DIR = '/content/drive/MyDrive/final-test/kcelectra_emo4_ft'
# ============================

from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import classification_report

# 1) 학습 데이터 구성 (시험지는 여전히 불변)
aug_map = {'base': None, '+voice': aug_voice, '+KOTE': aug_kote,
           '+voice+KOTE': pd.concat([aug_voice, aug_kote], ignore_index=True)}
aug = aug_map[FINAL_AUG]
train = tr_df[['text', 'label']].copy()
if aug is not None:
    train = pd.concat([train, aug[['text', 'label']]], ignore_index=True).drop_duplicates(subset='text')
train_y = train['label'].map(LAB2ID).values
print(f'최종 학습 조합: {FINAL_AUG} = {len(train):,}건')

# 2) 학습 (실험과 동일 설정)
tok = load_tokenizer(WINNER_MODEL)

class WT(Trainer):
    def __init__(self, class_weights=None, **kw):
        super().__init__(**kw); self.cw = class_weights
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop('labels')
        out = model(**inputs)
        loss = torch.nn.functional.cross_entropy(out.logits, labels, weight=self.cw)
        return (loss, out) if return_outputs else loss

class DS(torch.utils.data.Dataset):
    def __init__(self, texts, y):
        self.enc = tok(list(texts), padding=True, truncation=True, max_length=MAX_LEN)
        self.y = y
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        item['labels'] = torch.tensor(int(self.y[i])); return item

model = AutoModelForSequenceClassification.from_pretrained(
    MODELS[WINNER_MODEL], num_labels=len(EMO4)).to(device)
freq = pd.Series(train_y).value_counts().sort_index()
cw = torch.tensor((len(train_y) / (len(EMO4) * freq)).values, dtype=torch.float).to(device)
args = TrainingArguments(output_dir='/tmp/ft_final', num_train_epochs=3, learning_rate=2e-5,
    per_device_train_batch_size=32, per_device_eval_batch_size=64,
    fp16=(device == 'cuda'), logging_steps=200, seed=SEED,
    save_strategy='no', report_to='none')
trainer = WT(class_weights=cw, model=model, args=args, train_dataset=DS(train['text'].tolist(), train_y))
trainer.train()

# 3) 이중 평가 — 작성체 + 증량 150문장 (Drive에서 새로 로드)
chat150 = pd.DataFrame([json.loads(l) for l in
    open('/content/drive/MyDrive/final-test/chat_eval_set.jsonl', encoding='utf-8') if l.strip()])
c150_texts, c150_y = chat150['text'].tolist(), chat150['emotion'].map(LAB2ID).values
p_test = trainer.predict(DS(te_texts, te_y)).predictions.argmax(-1)
p_150 = trainer.predict(DS(c150_texts, c150_y)).predictions.argmax(-1)
print(f'\n[작성체 {len(te_texts):,}] Macro F1 {f1_score(te_y, p_test, average="macro"):.4f}')
print(f'[채팅체 {len(c150_texts)}문장 — 확정 평가] Macro F1 {f1_score(c150_y, p_150, average="macro"):.4f}')
print(classification_report(c150_y, p_150, target_names=EMO4, digits=3))

# 4) 저장 (Drive) — fp16 학습 후 텐서 재정렬 필수
for p in model.parameters():
    p.data = p.data.contiguous()
model.save_pretrained(SAVE_DIR); tok.save_pretrained(SAVE_DIR)
with open(SAVE_DIR + '/final_metrics.json', 'w', encoding='utf-8') as f:
    json.dump({'combo': FINAL_AUG, 'n_train': int(len(train)),
               'test_f1': float(f1_score(te_y, p_test, average='macro')),
               'chat150_f1': float(f1_score(c150_y, p_150, average='macro'))},
              f, ensure_ascii=False, indent=2)
print(f'\n모델 저장 완료 → {SAVE_DIR} (Drive에서 다운로드해 repo ai/emotion/artifacts_ft/ 에 배치)')

## 추가실험 1 — 게이트 임계값 보정 (자립형 · 로컬에서도 실행 가능)

모델 폴더와 chat_eval_set.jsonl만 있으면 어디서든 실행 가능 (GPU 불필요, CPU 2분).
서비스 `analysis_node`의 CONF_GATE(현행 0.55)를 새 모델 분포에 맞게 재보정한다.

In [ ]:
# ===== 추가실험 1: 확신도 게이트 임계값 보정 (자립형) =====
import json, numpy as np, torch
import torch.nn.functional as F
import pandas as pd
from transformers import AutoModelForSequenceClassification, AutoTokenizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
EMO4 = ['기쁨', '슬픔', '분노', '일반']
MDIR = '/content/drive/MyDrive/final-test/kcelectra_emo4_ft_voiceKOTE'   # 로컬이면 ai/emotion/artifacts_ft/... 로 변경
CHAT = '/content/drive/MyDrive/final-test/chat_eval_set.jsonl'           # 로컬이면 data/chat_eval_set.jsonl

gtok = AutoTokenizer.from_pretrained(MDIR)
gmodel = AutoModelForSequenceClassification.from_pretrained(MDIR).eval().to(device)
chat = pd.DataFrame([json.loads(l) for l in open(CHAT, encoding='utf-8') if l.strip()])
texts = chat['text'].tolist()
y = chat['emotion'].map({c: i for i, c in enumerate(EMO4)}).values

@torch.no_grad()
def predict_proba(ts, bs=64):
    out = []
    for i in range(0, len(ts), bs):
        e = gtok(ts[i:i+bs], padding=True, truncation=True, max_length=128,
                 return_tensors='pt').to(device)
        out.append(F.softmax(gmodel(**e).logits, dim=-1).cpu().numpy())
    return np.concatenate(out)

P = predict_proba(texts)
conf = P.max(1); pred = P.argmax(1); correct = (pred == y)
print(f'확신도: 평균 {conf.mean():.3f} / 중앙값 {np.median(conf):.3f}  (구 XGBoost 평균 0.667 → 게이트 0.55)')
print(f'정답일 때 {conf[correct].mean():.3f} vs 오답일 때 {conf[~correct].mean():.3f}')
print('\n임계값 | LLM 재분류 발동률 | 통과분 정확도 | 걸러진 것 정확도')
for th in [0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.95]:
    g = conf < th
    ka = correct[~g].mean() if (~g).sum() else float('nan')
    ga = correct[g].mean() if g.sum() else float('nan')
    print(f'  {th:.2f} |  {g.mean()*100:5.1f}%  |  {ka:.3f}  |  {ga:.3f}')
print('\n선택 기준: 발동률 10~25% + 걸러진 것의 정확도가 낮을 것')

## 추가실험 — 시드 반복 · 학습률 탐색 (승자 조합 고정)

전제: 위쪽 셀들(마운트→설치→로드→KOTE→학습기)이 실행된 상태.
- **시드 반복**: 승자 조합(+voice+KOTE)을 시드 3개로 학습 → 평균±표준편차 (오차범위 확보)
- **학습률 탐색**: 시드 42 고정, lr 3종 비교 (기본 2e-5는 시드 반복에서 이미 커버)

In [ ]:
# ===== 추가실험 2: 시드 반복 (+voice+KOTE, seeds 42/43/44) =====
# ⚠ 약 2시간 (40분 × 3) — 자리 비울 때 걸어둘 것
WINNER_AUG = pd.concat([aug_voice, aug_kote], ignore_index=True)

def build_train(aug):
    tr = pd.concat([tr_df[['text', 'label']], aug[['text', 'label']]],
                   ignore_index=True).drop_duplicates(subset='text')
    exam = {str(s).strip() for s in te_texts} | {str(s).strip() for s in chat_texts}
    tr = tr[~tr['text'].astype(str).str.strip().isin(exam)].reset_index(drop=True)
    return tr

seed_results = {}
for sd in [42, 43, 44]:
    print(f'===== seed {sd} =====')
    torch.manual_seed(sd); np.random.seed(sd); random.seed(sd)
    tr = build_train(WINNER_AUG)
    t, c = run_finetune(tr['text'].tolist(), tr['label'].map(LAB2ID).values)
    seed_results[sd] = {'test_f1': t['f1'], 'chat_f1': c['f1']}
    print(f"  작성체 {t['f1']:.4f} / 채팅체 {c['f1']:.4f}")

tf = [v['test_f1'] for v in seed_results.values()]
cf = [v['chat_f1'] for v in seed_results.values()]
print(f'\n작성체 F1: {np.mean(tf):.4f} ± {np.std(tf):.4f}')
print(f'채팅체 F1: {np.mean(cf):.4f} ± {np.std(cf):.4f}')
print('→ 계획서에 "평균±표준편차"로 기입 — 발표에서 오차범위 있는 유일한 팀이 될 것')

In [ ]:
# ===== 추가실험 3: 학습률 탐색 (+voice+KOTE, seed 42 고정) =====
# ⚠ 약 80분 (40분 × 2) — 2e-5는 시드 반복의 seed 42와 동일하므로 생략
lr_results = {}
for lr in [3e-5, 5e-5]:
    print(f'===== lr {lr} =====')
    torch.manual_seed(42); np.random.seed(42); random.seed(42)
    tr = build_train(WINNER_AUG)
    t, c = run_finetune(tr['text'].tolist(), tr['label'].map(LAB2ID).values, lr=lr)
    lr_results[lr] = {'test_f1': t['f1'], 'chat_f1': c['f1']}
    print(f"  작성체 {t['f1']:.4f} / 채팅체 {c['f1']:.4f}")
print('\n기본(2e-5, seed42) 대비 오르면 최종 모델 재학습 검토, 아니면 2e-5 확정')